In [1]:
from lucene_client import *
from datetime import datetime

ACCEPTED_SOURCES = ["WIKI", "WIKIDATA", "WIKT"]
API_SOURCES = ["WN", "OEWN", "MCR_ES", "OMWIKI"]

### SQL


In [ ]:
import sqlite3
from sqlite3 import IntegrityError
from languageprocessing import *
from OldApi.babelnet import *
import random


# Create (or connect to) a local database file
conn = sqlite3.connect("database.db")

# Create a cursor to execute SQL commands
cursor = conn.cursor()


In [ ]:

def create_tables():

    cursor.execute("DROP TABLE IF EXISTS words")
    cursor.execute("DROP TABLE IF EXISTS relations")
    cursor.execute("DROP TABLE IF EXISTS definitions")
    cursor.execute("DROP TABLE IF EXISTS dict")
    cursor.execute("DROP TABLE IF EXISTS gloss")

    # Scrapping
    cursor.execute("""
    CREATE TABLE IF NOT EXISTS dict (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        glossId TEXT NOT NULL UNIQUE,
        synsetId TEXT NOT NULL,
        type TEXT
    );
    """)

    cursor.execute("""
    CREATE TABLE IF NOT EXISTS lemma_synset(
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        lemma TEXT NOT NULL,
        language TEXT NOT NULL,
        synsetId TEXT NOT NULL,
        UNIQUE(lemma, language, synsetId)
    );
    """)

    cursor.execute("""
                   CREATE TABLE IF NOT EXISTS gloss (
                   id INTEGER PRIMARY KEY AUTOINCREMENT,
                   glossId TEXT NOT NULL,
                   glossLanguage TEXT NOT NULL,
                   glossSource TEXT NOT NULL,
                   glossNewId TEXT NOT NULL,
                   gloss TEXT NOT NULL                
                   );
                   """)

    # Create a table
    cursor.execute("""
    CREATE TABLE IF NOT EXISTS words (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        word TEXT NOT NULL,  
        language TEXT NOT NULL,
        concept TEXT,
        scrapped BOOLEAN DEFAULT FALSE,
        translated BOOLEAN DEFAULT FALSE
    )
    """)

    # Enforce uniqueness when concept IS set
    cursor.execute("""
    CREATE UNIQUE INDEX IF NOT EXISTS uniq_word_lang_concept
    ON words (word, language, concept)
    WHERE concept IS NOT NULL
    """)

    # Enforce uniqueness when concept IS NULL (only one such row per word+language)
    cursor.execute("""
    CREATE UNIQUE INDEX IF NOT EXISTS uniq_word_lang_null_concept
    ON words (word, language)
    WHERE concept IS NULL
    """)

    cursor.execute("""
    CREATE TABLE IF NOT EXISTS relations (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        word1_id INTEGER NOT NULL,  
        word2_id INTEGER NOT NULL,
        relation_type TEXT NOT NULL,
        FOREIGN KEY (word1_id) REFERENCES words(id),
        FOREIGN KEY (word2_id) REFERENCES words(id),
        UNIQUE(word1_id, word2_id, relation_type)
    )
    """)


    cursor.execute("""
    CREATE TABLE IF NOT EXISTS definitions (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        synset_id INTEGER NOT NULL,
        definition TEXT NOT NULL,
        source TEXT NOT NULL,
        UNIQUE(synset_id, definition)
    )
    """)

    conn.commit()
    print("Table created successfully!")


def drop_tables():
    cursor.execute("DROP TABLE IF EXISTS words")
    cursor.execute("DROP TABLE IF EXISTS relations")
    cursor.execute("DROP TABLE IF EXISTS definitions")
    cursor.execute("DROP TABLE IF EXISTS dict")
    cursor.execute("DROP TABLE IF EXISTS gloss")
    cursor.execute("DROP TABLE IF EXISTS lemma_synset")
    conn.commit()
    print("Tables dropped successfully!")


def transform_doc_to_word(doc):
    word = {}
    word['glossId'] = doc.get('glossId', [None])[0]
    word['synsetId'] = doc.get('synsetId', [None])[0]
    word["type"] = doc.get('type', [None])[0]
    return word

def insert_concept(doc):
    cursor.execute("""
    INSERT INTO dict (
        glossId,
        synsetId,
        type
    )
    VALUES (
        :glossId,
        :synsetId,
        :type
    )
    """, doc)
    conn.commit()

def insert_lemma_synset(lemma, language, synsetId):
    cursor.execute("""
    INSERT INTO lemma_synset (
        lemma,
        language,
        synsetId
    )
    VALUES (
        :lemma,
        :language,
        :synsetId
    )
    """, {'lemma': lemma, 'language': language, 'synsetId': synsetId})
    conn.commit()

def insert_gloss(doc, gloss):

    gloss['glossId'] = doc['glossId']
    cursor.execute("""
    INSERT INTO gloss (
        glossId,
        glossLanguage,
        glossSource,
        glossNewId,
        gloss
    )
    VALUES (
        :glossId,
        :glossLanguage,
        :glossSource,
        :glossNewId,
        :gloss
    )
    """, gloss)

    conn.commit()



def insert_word_and_definition(synset_id, word, definition, language, source, scrapped=False, translated=False):
    definition_words = preprocess_text(definition, language=language)["lemmatized"]
    cursor.execute(
    """
    INSERT INTO words (word, language, concept) VALUES (?, ?, ?)
    """, (word, language, synset_id))

    try:
        word_id = cursor.lastrowid
        cursor.execute("INSERT INTO definitions (synset_id, definition, source) VALUES (?, ?, ?)", (synset_id, definition, source))
    except IntegrityError:
        pass

    
    definition_word_ids = []

    for def_word in definition_words:
        cursor.execute(
            "SELECT id FROM words WHERE word = ? AND language = ?",
            (def_word, language)
        )
        existing = cursor.fetchone()

        if existing:
            definition_word_ids.append(existing[0])
        else:
            cursor.execute(
                "INSERT INTO words (word, language, concept, scrapped) VALUES (?, ?, ?, ?)",
                (def_word, language, None, scrapped)
            )
            definition_word_ids.append(cursor.lastrowid)

    relations = [
        (word_id, def_word_id, "definition")
        for def_word_id in definition_word_ids
    ]

    cursor.executemany(
        "INSERT INTO relations (word1_id, word2_id, relation_type) VALUES (?, ?, ?)",
        relations
    )

def insert_words_and_definition(synset_id, words, definition, language, source, scrapped=False, translated=False):
    definition_words = preprocess_text(definition, language=language)["lemmatized"]

    # Insert the definition once (shared by all words in this synset)
    try:
        cursor.execute(
            "INSERT INTO definitions (synset_id, definition, source) VALUES (?, ?, ?)",
            (synset_id, definition, source)
        )
    except IntegrityError:
        pass

    # Resolve/insert definition words once, reused for every word
    definition_word_ids = []
    for def_word in definition_words:
        cursor.execute(
            "SELECT id FROM words WHERE word = ? AND language = ?",
            (def_word, language)
        )
        existing = cursor.fetchone()

        if existing:
            definition_word_ids.append(existing[0])
        else:
            cursor.execute(
                "INSERT INTO words (word, language, concept, scrapped) VALUES (?, ?, ?, ?)",
                (def_word, language, None, scrapped)
            )
            definition_word_ids.append(cursor.lastrowid)

    # Insert each word and link it to the definition words
    relations = []
    word_ids = []
    for word in words:
        try:
            cursor.execute(
                "INSERT INTO words (word, language, concept) VALUES (?, ?, ?)",
                (word, language, synset_id)
            )
            word_id = cursor.lastrowid
            word_ids.append(word_id)

            relations.extend(
                (word_id, def_word_id, "definition")
                for def_word_id in definition_word_ids
            )
        except IntegrityError:
            pass


    cursor.executemany(
        "INSERT OR IGNORE INTO relations (word1_id, word2_id, relation_type) VALUES (?, ?, ?)",
        relations
    )

def insert_translation_relations(synset_id):
    # Get all word ids already linked to this synset (the "original" words)
    cursor.execute(
        "SELECT id FROM words WHERE language = 'EN' AND concept = ?",
        (synset_id,)
    )
    en_word_ids = [row[0] for row in cursor.fetchall()]

    cursor.execute(
        "SELECT id FROM words WHERE language = 'ES' AND concept = ?",
        (synset_id,)
    )
    es_word_ids = [row[0] for row in cursor.fetchall()]

    relations = [
        (orig_id, trans_id, "translation")
        for orig_id in en_word_ids
        for trans_id in es_word_ids
    ]

    cursor.executemany(
        "INSERT OR IGNORE INTO relations (word1_id, word2_id, relation_type) VALUES (?, ?, ?)",
        relations
    )

In [4]:
def get_dict_from_glossId(glossId):
    cursor.execute("SELECT * FROM dict WHERE glossId = ?", (glossId,))
    return cursor.fetchall()

def get_dict_from_synsetId(synsetId):
    cursor.execute("SELECT * FROM dict WHERE synsetId = ?", (synsetId,))
    return cursor.fetchall()

def get_glosses_from_glossId(glossId):
    cursor.execute("SELECT * FROM gloss WHERE glossId = ?", (glossId,))
    return cursor.fetchall()

def get_lemma_synset_from_synsetId(synsetId):
    cursor.execute("SELECT * FROM lemma_synset WHERE synsetId = ?", (synsetId,))
    return cursor.fetchall()

def get_glosses_from_synsetId(synsetId):
    cursor.execute("SELECT * FROM gloss WHERE glossId IN (SELECT glossId FROM dict WHERE synsetId = ?)", (synsetId,))
    return cursor.fetchall()

def get_definitions_from_word(word):
    cursor.execute("SELECT * FROM definitions WHERE synset_id IN (SELECT concept FROM words WHERE word = ?)", (word,))
    return cursor.fetchall()

def get_word_id_from_synset_language_word(synset_id, language, word):
    cursor.execute("SELECT id FROM words WHERE concept = ? AND language = ? AND word = ?", (synset_id, language, word))
    result = cursor.fetchall()
    return result

def get_word_from_concept(concept):
    cursor.execute("SELECT * FROM words WHERE concept = ?", (concept,))
    return cursor.fetchall()

def get_relations_between_words(word1, word2):
    cursor.execute("""
    SELECT * FROM relations 
    WHERE word1_id IN (SELECT id FROM words WHERE word = ?) 
    AND word2_id IN (SELECT id FROM words WHERE word = ?)
    """, (word1, word2))
    rows = cursor.fetchall()
    cursor.execute("""
    SELECT * FROM relations 
    WHERE word1_id IN (SELECT id FROM words WHERE word = ?) 
    AND word2_id IN (SELECT id FROM words WHERE word = ?)
    """, (word2, word1))
    rows.extend(cursor.fetchall())
    return rows

def get_relations_for_word(word):
    cursor.execute("""
    SELECT * FROM relations 
    WHERE word1_id IN (SELECT id FROM words WHERE word = ?) 
    OR word2_id IN (SELECT id FROM words WHERE word = ?)
    """, (word, word))
    rows = cursor.fetchall()
    return rows

def get_translation_relations_for_word(word):
    cursor.execute("""
    SELECT * FROM relations 
    WHERE relation_type = 'translation' AND (word1_id IN (SELECT id FROM words WHERE word = ?) 
    OR word2_id IN (SELECT id FROM words WHERE word = ?))
    """, (word, word))
    rows = cursor.fetchall()
    return rows

def set_translation_between_wordsIds(word1_id, word2_id):
    cursor.execute("""
    INSERT OR IGNORE INTO relations (word1_id, word2_id, relation_type) VALUES (?, ?, ?)
    """, (word1_id, word2_id, "translation"))
    conn.commit()

def get_single_word_id(word, concept, language):
    cursor.execute("SELECT id FROM words WHERE word = ? AND language = ? AND concept = ?", (word, language, concept))
    result = cursor.fetchone()
    return result[0] if result else None

def get_word_from_id(word_id):
    cursor.execute("SELECT word, language, concept FROM words WHERE id = ?", (word_id,))
    result = cursor.fetchone()
    if result:
        return {"word": result[0], "language": result[1], "concept": result[2]}
    else:
        return None

# Srapping Functions


In [5]:
def scrap_word(word, language):
    """
    Searches all the different concepts for a given word in a given language and stores them in the database, along with the relations between the word and the definition in its definition.

    Parameters:
        word (str): the word to search for
        language (str ["EN", "ES"]): the language of the word

    Returns:
        None
    """
    cursor.execute("SELECT * FROM words WHERE word = ? AND language = ?", (word, language))
    word_exists = cursor.fetchone()
    if word_exists and word_exists[4]: # scrapped == True
        print(f"Word '{word}' in language '{language}' already scrapped, skipping.")
        return
    
    request = search_word(word, lang=language)
    print(request)
    ids = [id["id"] for id in request]
    cursor.execute("SELECT concept FROM words WHERE word = ? AND language = ?", (word, language))
    existing_ids = cursor.fetchall()
    existing_ids = set([id[0] for id in existing_ids])
    number_of_ids = len(ids)
    for existing_id in existing_ids:
        if (existing_id in ids):
            ids.remove(existing_id)
    print(number_of_ids - len(ids), "existing ids removed")
    for id in range(len(ids)):
        #print(f"Scrapping {word} {ids[id]} ({language}) {id+1}/{len(ids)}")
        try:
            get_syn = get_synset(ids[id], lang=language)
            

            if "NAMED_ENTITY" in get_syn.get("tags", []) or not get_syn.get("glosses"):
                continue
            if (get_syn.get("senses") == [] or "NAMED_ENTITY" in get_syn.get("senses", [{}])[0].get("properties", {}).get("tags", [])):
                continue

            word_set = set()
            for sense in get_syn.get("senses", []):
                if sense.get("properties", {}).get("source") in API_SOURCES and sense.get("properties", {}).get("lemma", {}).get("type") == "HIGH_QUALITY" and sense.get("properties", {}).get("language") == language:
                    lemma = sense.get("properties", {}).get("lemma", {}).get("lemma")
                    if "_" in lemma or "(" in lemma or ")" in lemma or "-" in lemma:
                        continue
                    word_set.add(lemma)
            if len(word_set) == 0:
                continue
            for gloss in range(len(get_syn["glosses"])):
                if get_syn["glosses"][gloss]["language"] == language:
                    if get_syn["glosses"][gloss]["source"] in ["WN", "OEWN", "MCR_ES", "OMWIKI"]:
                        ids_definition = get_syn["glosses"][gloss]["gloss"]
                        for lemma in word_set:
                            insert_words_and_definition(synset_id=ids[id], words=word_set, definition=ids_definition, source=get_syn["glosses"][gloss]["source"], language=language, scrapped=False)

            else:
                #print(f"Synset {ids[id]} for word {word} did not meet the conditions and was skipped.")
                pass
        except Exception as e:
            #print(f"Error occurred while processing synset {ids[id]} for word {word}: {e}")
            continue
    cursor.execute("UPDATE words SET scrapped = 1 WHERE word = ? AND language = ?", (word, language))
    conn.commit()


In [6]:
def scrap_translations(synset_id, language):
    cursor.execute("SELECT * FROM words WHERE concept = ? AND language = ? AND translated = 0", (synset_id, language))
    word_exists = cursor.fetchone()
    if word_exists and word_exists[5]: # translated == True
        print(f"concept '{synset_id}' in language '{language}' already translated, skipping.")
        return
    synonyms = get_synset_synonyms(synset_id, lang=language)
    synonyms = list(set(synonyms))
    #print(synonyms)
    synset_data = get_synset(synset_id, lang=language)
    glosses = synset_data.get('glosses', [])
    for gloss in glosses:
        if gloss['language'] == language and apply_condition(gloss, ["WN", "OEWN", "MCR_ES", "OMWIKI", "WIKIDATA"], ["source"]):
            try:
                print(gloss)
                definition = gloss['gloss']
                #print(synonym, synset_id)
                insert_words_and_definition(synset_id=synset_id, words=synonyms, definition=definition, source=gloss['source'], language=language)
                insert_translation_relations(synset_id)
                break
            except Exception as e:
                print(f"Error occurred while inserting translation for {synset_id}: {e}")
                continue
    cursor.execute("UPDATE words SET translated = 1 WHERE concept = ? AND language = ?", (synset_id, language))
    conn.commit()

In [7]:

def recursive_translations():
    cursor.execute("SELECT DISTINCT w.language, w.concept FROM words w WHERE w.translated IS FALSE AND w.concept IS NOT NULL AND NOT EXISTS (SELECT 1 FROM words w2 WHERE w2.concept = w.concept AND w2.language != w.language)")
    rows = cursor.fetchall()
    i=1
    random.shuffle(rows)
    for language, concept in rows:
        print(f"{i}/{len(rows)}")
        i += 1
        lang = "ES" if language == "EN" else "EN"
        scrap_translations(concept, lang)


def recursive_scrap():
    print("Starting recursive scrap...")
    cursor.execute("SELECT word, language FROM words WHERE NOT EXISTS (SELECT 1 FROM words w2 WHERE w2.word = words.word AND w2.language = words.language AND w2.scrapped = 1)")
    rows = cursor.fetchall()
    print(len(rows))
    random.shuffle(rows)
    i = 1
    for word, language in rows:
        print(f"{i}/{len(rows)}")
        scrap_word(word, language)
        i += 1

In [8]:
def scrap_word_local(word):
    results = dict_search(f"LEMMA:{word}", max_results=100000)["results"]
    synset_id_scrapped = set()
    for result in results:
        lemmaLanguagues = result.get("languageLemma", None)
        synsetId = result["synsetId"][0]
        glossId = result["glossId"]
        AcceptedLemmaLanguages = []
        for lemmaLanguague in lemmaLanguagues:
            lemmaLanguage = lemmaLanguague.split(":")
            if lemmaLanguage[0] not in ["EN", "ES"]:
                continue
            npl_lemma = preprocess_text(lemmaLanguage[1], language=lemmaLanguage[0])["lemmatized"]
            if npl_lemma is not None and len(npl_lemma) == 1:
                lemmaLanguage[1] = npl_lemma[0]
            else:
                continue
            if lemmaLanguage[0] == "EN" or lemmaLanguage[0] == "ES":
                language = lemmaLanguage[0]
                lemma = lemmaLanguage[1]
                if result["type"][0] == "NAMED_ENTITY":
                    continue
                if "_" in lemma or "(" in lemma or ")" in lemma or "-" in lemma:
                    continue
                if not lemma.isalpha():
                    continue
                cursor.execute("SELECT * FROM words WHERE word = ? AND language = ?", (lemma, language))
                existing_words = cursor.fetchall()
                for existing_word in existing_words:
                    if existing_word[3] is not None and existing_word[3] == result["synsetId"][0]:
                        #print(f"Word '{existing_word[1]}' in language '{existing_word[2]}' already has the concept '{existing_word[3]}' as {result['synsetId']}. Skipping update.")
                        continue
                AcceptedLemmaLanguages.append([lemma, language])
        doc = transform_doc_to_word(result)
        word_in_lemma = False
        for lemma, language in AcceptedLemmaLanguages:
            if word == lemma:
                word_in_lemma = True
                break
        if not word_in_lemma: # Si la palabra no está en los lemas aceptados, no se inserta el concepto ni los lemas, dado que puede ser una palabra parecida pero no la misma.
            continue
        try:
            insert_concept(doc)
            for lemma, language in AcceptedLemmaLanguages:
                insert_lemma_synset(lemma, language, synsetId)
        except IntegrityError as e:
            #print(f"Error occurred while inserting concept: {e}")
            pass
                
        
        glosses = gloss_search(glossId, max_results=100000)["results"]
        if glosses is None or len(glosses) == 0:
            continue
        glosses = glosses[0]
        for gloss in glosses["gloss"]:
            gloss_dict = gloss_parser(gloss)
            if gloss_dict is None:
                continue
            if gloss_dict["glossLanguage"] != "ES" and gloss_dict["glossLanguage"] != "EN":
                continue
            if gloss_dict["glossSource"] not in ACCEPTED_SOURCES:
                continue 
            try:
                insert_gloss(doc, gloss_dict)
                synset_id_scrapped.add(synsetId)
            except IntegrityError as e:
                #print(f"Error occurred while inserting gloss: {e}")
                pass  # or continue to whatever outer loop this belongs to
            for lemmaLanguage in AcceptedLemmaLanguages:
                lemma, language = lemmaLanguage
                if language == gloss_dict["glossLanguage"]:
                    try:
                        insert_word_and_definition(
                            synsetId,
                            lemma,
                            definition=gloss_dict["gloss"],
                            language=gloss_dict["glossLanguage"],
                            source=gloss_dict["glossSource"],
                        )
                    except IntegrityError as e:
                        #print(f"Error occurred while inserting word/definition: {e}")
                        continue
    for synset_id in synset_id_scrapped:
        cursor.execute("UPDATE words SET scrapped = 1 WHERE concept = ?", (synset_id,))
    cursor.execute("UPDATE words SET scrapped = 1 WHERE word = ?", (word,))


In [9]:
def recursive_words_local():
    cursor.execute("SELECT DISTINCT word, language, concept FROM words WHERE scrapped = 0 AND concept IS NULL")
    rows = cursor.fetchall()
    result = [{"word": row[0], "language": row[1], "concept": row[2]} for row in rows]
    i = 1
    for row in result:
        print(f"{i}/{len(result)}")
        i += 1
        scrap_word_local(row["word"])
        

In [10]:
def scrap_translation_local(concept):
    ### SI LA TRADUCCIÓN NO EXISTE, SE BUSCA EN LUCENE
    requests = dict_search(f'SYNSET_ID:*{concept}', max_results=100000)
    for result in requests['results']:
        lemmaLanguagues = result.get("languageLemma", None)
        synsetId = result["synsetId"][0]
        if not synsetId.endswith(concept) : ### Making sure we are only looking at the concept we want to translate
            continue 
        for lemmaLanguague in lemmaLanguagues:
            lemmaLanguage = lemmaLanguague.split(":")
            if lemmaLanguage[0] in ["EN", "ES"]:
                scrap_word_local(lemmaLanguage[1])
            

In [11]:
def recursive_translations_local():
    cursor.execute("SELECT DISTINCT concept FROM words WHERE translated = 0 AND concept IS NOT NULL")
    results = cursor.fetchall()
    words = [{"concept": row[0]} for row in results]
    i=1
    for word in words:
        try:
            print(f"{i}/{len(words)}")
            i += 1
            concept = word["concept"]
            concept = concept.split(":")[-1]  # Extract the actual concept ID from the synset ID
            scrap_translation_local(concept)
            cursor.execute("UPDATE words SET translated = 1 WHERE concept = ?", (concept,))
        except Exception as e:
            print(f"Error occurred while processing concept {concept}: {e}")
            continue

In [12]:
def make_relations():
    cursor.execute("""
        INSERT OR IGNORE INTO relations (word1_id, word2_id, relation_type)
        SELECT w1.id, w2.id, 'translation'
        FROM words w1
        JOIN words w2
            ON w1.concept = w2.concept
            AND w1.id != w2.id
            AND w1.language != w2.language
        WHERE w1.concept IS NOT NULL
    """)
    conn.commit()


In [13]:
def database_backup(cycle_number, cycle_state):
    # Create a backup of the database
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    db_bame = f"{timestamp}_database_backup_{cycle_number}_{cycle_state}.db"
    backup_conn = sqlite3.connect(db_bame)
    with backup_conn:
        conn.backup(backup_conn)
    backup_conn.close()
    print(f"Database '{db_bame}' backup created successfully.")

def local_cycle():
    cycle_number = 1
    
    while True:
        try:
            recursive_words_local()
            make_relations()

            database_backup(cycle_number=cycle_number, cycle_state="local_cycle")

            cycle_number += 1
        except Exception as e:
            print(f"Error occurred during local cycle: {e}")
            break

def api_cycle():
    cycle_number = 1

    while True:
        try:
            recursive_scrap()
            make_relations()
            recursive_translations()
            make_relations()
            database_backup(cycle_number=cycle_number, cycle_state="api_cycle")

            cycle_number += 1
        except Exception as e:
            print(f"Error occurred during API cycle: {e}")
            break

# Testing

In [ ]:
import random

def test_local(sample_size=1000):
    cursor.execute("SELECT DISTINCT word, language FROM words where concept IS NOT NULL")
    rows = cursor.fetchall()
    sample = random.sample(rows, sample_size)
    print("Sample of words from the database:")
    corrects = 0

    target_language = {"EN": "ES", "ES": "EN"}

    def end_cicle(definition_relation_not_found, translation_relation_not_found):
        if len(definition_relation_not_found) > 0:
            return False
        if len(translation_relation_not_found) > 0:
            return False
        return True
    
    for word in sample:
        definition_relation_not_found = []
        translation_relation_not_found = []


        definitions = dict_search(f"LEMMA:{word[0]}", max_results=100000)["results"]
        if len(definitions) == 0:
            #print(f"No definitions found for word: {word[0]}")
            if end_cicle(definition_relation_not_found, translation_relation_not_found):
                corrects += 1
            continue
        for definition in definitions:

            glosses = gloss_search(definition["glossId"], max_results=100000)["results"]

            if len(glosses) == 0:
                continue
            for gloss in glosses[0]["gloss"]:
                gloss_dict = gloss_parser(gloss)
                if gloss_dict["glossLanguage"] == word[1]:
                    glossText = preprocess_text(gloss_dict["gloss"], language=word[1])["lemmatized"]
                    #print(f"Word: {word[0]}, Language: {word[1]}, Gloss: {glossText}")

                    relations = get_relations_for_word(word[0])
                    for relation in relations:
                        id1 = relation[1] # word we are checking
                        id2 = relation[2] # second word, that should be in the gloss
                        if get_word_from_id(id2)["word"] == word[0]: # means this relation, the word we are checking is being used to define another word.
                            continue
                        cursor.execute("SELECT word FROM words WHERE id = ?", (id2,))
                        word2 = cursor.fetchone()[0]

                        if word[0] == word2:
                            if word2 in glossText:
                                if word2 in definition_relation_not_found:
                                    definition_relation_not_found.remove(word2)
                                #print(f"Relation found: {word} - {word2} ({relation[3]})")
                            else:
                                definition_relation_not_found.append((word[0], word2))


            translations = get_translation_relations_for_word(word[0])
            translation_relations = []
            for translation in translations:
                id1 = translation[1]
                id2 = translation[2]
                word1 = get_word_from_id(id1)["word"]
                word2 = get_word_from_id(id2)["word"]
                translation_relations.append((word1, word2))
            for lemmaLanguague in definition.get("languageLemma", []):
                lemmaLanguage = lemmaLanguague.split(":")
                language = lemmaLanguage[0]
                lemma = lemmaLanguage[1]
                if definition["type"][0] == "NAMED_ENTITY":
                    continue
                if "_" in lemma or "(" in lemma or ")" in lemma or "-" in lemma:
                    continue
                if not lemma.isalpha():
                    continue
                if language == target_language[word[1]]:
                    if (word[0], lemma) in translation_relations or (lemma, word[0]) in translation_relations:
                        if (word[0], lemma) in translation_relation_not_found:
                            translation_relation_not_found.remove((word[0], lemma))
                        #print(f"Translation relation found: {word} - {lemma}")
                    else:
                        translation_relation_not_found.append((word[0], lemma))

            
        if end_cicle(definition_relation_not_found, translation_relation_not_found):
            corrects += 1
        else:
            print(f"Relations not found for word: {word[0]}, Language: {word[1]}")
            if len(definition_relation_not_found) > 0:
                print(f"Definition relations not found: {definition_relation_not_found}")
            if len(translation_relation_not_found) > 0:
                print(f"Translation relations not found: {translation_relation_not_found}")
    print(f"Correctly found relations for {corrects}/{sample_size} words.")

#test_local(sample_size=10000)

# Scrapping


In [15]:
with open("../ogden.txt", "r") as f:
    text = f.read()


text = text.replace("\n", " ")
text = preprocess_text(text, language="EN")["lemmatized"]
text = " ".join(text)
print(text)
words = text.split(" ")
for i in range(len(words)):
    words[i] = words[i].strip()

able account acid across act addition adjustment advertisement agreement air almost among amount amusement angle angry animal answer ant apparatus apple approval arch argument arm army art attack attempt attention attraction authority automatic awake baby back bad bag balance ball band base basin basket bath beautiful bed bee behavior belief bell bent berry bird birth bit bite bitter black blade blood blow blue board boat body boiling bone book boot bottle box boy brain brake branch brass bread breath brick bridge bright break brother brown brush bucket building bulb burn burst business butter button cake camera canvas card care carriage cart cat cause certain chain chalk chance change cheap cheese chemical chest chief chin church circle clean clear clock cloth cloud coal coat cold collar color comb come comfort committee common company comparison complete competition complex condition connection conscious control cook copper copy cord cork cotton cough country cover cow crack credit c

### Local Scrapping

In [16]:
# conn = sqlite3.connect("database.db")
# cursor = conn.cursor()
# drop_tables()
# create_tables()
# i = 1
# for word in words:
#     print(f"{i}/{len(words)}")
#     i += 1
#     scrap_word_local(word)
# make_relations() 
# database_backup(cycle_number=0, cycle_state="local_ogden_cycle")


# while True:
#     try:
#         local_cycle()
#     except Exception as e:
#         print(f"Error occurred during local cycle: {e}")
#         break

### API Scrapping


In [17]:
conn = sqlite3.connect("api_database.db")
cursor = conn.cursor()


database_backup(cycle_number=1, cycle_state="api_cycle_end")

Database '20260718_052342_database_backup_1_api_cycle_end.db' backup created successfully.


In [ ]:
# import random

# random.seed(10) # Seed = 10 

# sample = random.sample(words, 10)

# conn = sqlite3.connect("api_database.db")
# cursor = conn.cursor()



# # # drop_tables()
# # # create_tables()
# # j = 1
# # for word in words:
# #     print(f"{j}/{len(words)}")
# #     j += 1
# #     scrap_word(word, "EN")


# database_backup(cycle_number=1, cycle_state="api_cycle_start")
# make_relations()
# recursive_translations()
# make_relations()

# # # recursive_translations()
# while True:
#     try:
#         api_cycle()
#     except Exception as e:
#         print(f"Error occurred during API cycle: {e}")
#         database_backup(cycle_number=1, cycle_state="api_cycle_start")
#         break